# Seasonal & Price Analysis — Weekly Exploration

**Purpose:** Review weekly price structure (MA alignment, Donchian channels) and seasonal patterns for all 4 markets. Combine with COT context from notebook 01 to build the full weekly picture.

**Workflow:** Run after `01_cot_exploration.ipynb`. The price charts show *where* the market is; the seasonal charts show *what usually happens* this week of year; COT shows *who is positioned how*. The combination is the story.

**Markets:** NQ, GC, CL, ZC

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

from src.data.prices import build_price_dataset, build_weekly_dataset
from src.data.config import MARKETS
from src.models.seasonal import compute_seasonal_matrix
from src.viz.charts import chart_seasonal, chart_price_daily, chart_price_weekly

%matplotlib inline
pd.set_option('display.float_format', '{:.2f}'.format)

SYMBOLS = ['NQ', 'GC', 'CL', 'ZC']
current_week = date.today().isocalendar()[1]
print(f"Report date: {date.today().strftime('%B %d, %Y')} | Week {current_week}")

## 1. Load Price Data

In [ ]:
FORCE_REFRESH = False  # flip to True to re-download from yfinance

prices = {}
for sym in SYMBOLS:
    prices[sym] = build_price_dataset(sym, force_refresh=FORCE_REFRESH)
    last = prices[sym].iloc[-1]
    print(f"{sym:>2}: {last['close']:>10,.2f}  |  "
          f"SMA20: {last['sma_20']:>10,.2f}  |  "
          f"SMA50: {last['sma_50']:>10,.2f}  |  "
          f"SMA200: {last['sma_200']:>10,.2f}  |  "
          f"ATR14: {last['atr_14']:>8,.2f}")

## 2. MA Structure Snapshot

Quick read on trend alignment for each market. The table shows where price sits relative to each MA and how far (in ATR units) — useful for gauging how extended or mean-reverting a market is.

**Reading the table:**
- **% from MA** — positive = price above MA (bullish), negative = below (bearish)
- **ATR dist** — distance in ATR units. >2 ATR from a key MA = extended, likely to revert

In [ ]:
rows = []
for sym in SYMBOLS:
    last = prices[sym].iloc[-1]
    close = last['close']
    atr = last['atr_14']
    
    row = {'Market': sym, 'Close': close, 'ATR14': atr}
    for ma in [20, 50, 100, 200]:
        ma_val = last[f'sma_{ma}']
        pct = ((close / ma_val) - 1) * 100
        atr_dist = (close - ma_val) / atr
        row[f'{ma}d %'] = pct
        row[f'{ma}d ATR'] = atr_dist
    rows.append(row)

ma_df = pd.DataFrame(rows)
ma_df.style.format({
    'Close': '{:,.2f}', 'ATR14': '{:,.2f}',
    '20d %': '{:+.1f}%', '50d %': '{:+.1f}%', '100d %': '{:+.1f}%', '200d %': '{:+.1f}%',
    '20d ATR': '{:+.1f}', '50d ATR': '{:+.1f}', '100d ATR': '{:+.1f}', '200d ATR': '{:+.1f}',
}).background_gradient(
    subset=['200d %'], cmap='RdYlGn', vmin=-15, vmax=15
)

## 3. Donchian Channel Position

Where is price within the 50-day Donchian channel? This tells you if we're near breakout highs, breakdown lows, or mid-range.

- **>80%** = near the top of the channel (bullish momentum or overbought)
- **<20%** = near the bottom (bearish momentum or oversold)
- **40-60%** = mid-range, no strong directional read

In [ ]:
dc_rows = []
for sym in SYMBOLS:
    last = prices[sym].iloc[-1]
    close = last['close']
    
    for period in [20, 50]:
        upper = last[f'donchian_{period}_upper']
        lower = last[f'donchian_{period}_lower']
        width = upper - lower
        position = ((close - lower) / width * 100) if width > 0 else 50
        
        dc_rows.append({
            'Market': sym,
            'Channel': f'{period}d',
            'Upper': upper,
            'Lower': lower,
            'Close': close,
            'Position %': position,
        })

dc_df = pd.DataFrame(dc_rows)
dc_df.style.format({
    'Upper': '{:,.2f}', 'Lower': '{:,.2f}', 'Close': '{:,.2f}',
    'Position %': '{:.0f}%',
}).background_gradient(
    subset=['Position %'], cmap='RdYlGn', vmin=0, vmax=100
)

## 4. Seasonal Snapshot — This Week Across All Markets

How does the current week-of-year typically perform? Compare across lookback windows (5yr, 10yr, 20yr) to see if the pattern is consistent or noisy.

In [ ]:
lookbacks = [5, 10, 20]

rows = []
for sym in SYMBOLS:
    sm = compute_seasonal_matrix(sym, lookbacks=lookbacks)
    
    if current_week not in sm.index:
        continue
    
    row = {'Market': sym, 'Week': current_week}
    for lb in lookbacks:
        label = f'{lb}yr'
        row[f'Ret {label}'] = sm.loc[current_week, f'mean_return_{label}'] * 100
        row[f'WR {label}'] = sm.loc[current_week, f'win_rate_{label}']
        row[f'Std {label}'] = sm.loc[current_week, f'std_{label}'] * 100
    rows.append(row)

seasonal_df = pd.DataFrame(rows)
seasonal_df.style.format({
    'Week': '{:.0f}',
    'Ret 5yr': '{:+.2f}%', 'WR 5yr': '{:.0f}%', 'Std 5yr': '{:.2f}%',
    'Ret 10yr': '{:+.2f}%', 'WR 10yr': '{:.0f}%', 'Std 10yr': '{:.2f}%',
    'Ret 20yr': '{:+.2f}%', 'WR 20yr': '{:.0f}%', 'Std 20yr': '{:.2f}%',
}).background_gradient(
    subset=[c for c in seasonal_df.columns if c.startswith('WR')],
    cmap='RdYlGn', vmin=20, vmax=80
)

## 5. Next 4 Weeks — Seasonal Look-Ahead

What's coming up seasonally? Look at the next 4 weeks to spot windows of opportunity or risk. High win rate + positive return across multiple lookbacks = strong seasonal setup.

In [ ]:
weeks_ahead = range(current_week, current_week + 5)

for sym in SYMBOLS:
    sm = compute_seasonal_matrix(sym, lookbacks=[10])
    label = '10yr'
    
    print(f"\n{'='*50}")
    print(f"  {sym} — Weeks {current_week} to {current_week + 4}")
    print(f"{'='*50}")
    print(f"  {'Week':>4}  {'Return':>8}  {'WR':>5}  {'Std':>7}  Signal")
    print(f"  {'----':>4}  {'------':>8}  {'--':>5}  {'---':>7}  ------")
    
    for w in weeks_ahead:
        wk = w if w <= 53 else w - 53
        if wk not in sm.index:
            continue
        ret = sm.loc[wk, f'mean_return_{label}'] * 100
        wr = sm.loc[wk, f'win_rate_{label}']
        std = sm.loc[wk, f'std_{label}'] * 100
        
        # Simple signal logic
        if wr >= 70 and ret > 0:
            signal = "BULLISH"
        elif wr <= 30 and ret < 0:
            signal = "BEARISH"
        elif wr >= 60 and ret > 0:
            signal = "lean bull"
        elif wr <= 40 and ret < 0:
            signal = "lean bear"
        else:
            signal = "—"
        
        marker = " ◀ current" if wk == current_week else ""
        print(f"  W{wk:>2}   {ret:>+7.2f}%  {wr:>4.0f}%  {std:>6.2f}%  {signal}{marker}")

---

## 6. NQ — E-mini Nasdaq 100

Key levels to watch: SMA 200 (long-term trend), SMA 50 (medium-term momentum), Donchian channel edges (breakout/breakdown).

In [ ]:
chart_price_daily('NQ', save=False)
plt.show()
chart_price_weekly('NQ', save=False)
plt.show()
chart_seasonal('NQ', save=False)
plt.show()

**NQ Notes:**

- Trend structure (MAs aligned or crossing?):
- Distance from key MAs:
- Seasonal bias this week:
- Combined read (price + seasonal + COT from nb01):

---

## 7. GC — Gold

In [ ]:
chart_price_daily('GC', save=False)
plt.show()
chart_price_weekly('GC', save=False)
plt.show()
chart_seasonal('GC', save=False)
plt.show()

**GC Notes:**

- Trend structure:
- Extension from MAs (overheated?):
- Seasonal bias:
- Combined read:

---

## 8. CL — WTI Crude Oil

In [ ]:
chart_price_daily('CL', save=False)
plt.show()
chart_price_weekly('CL', save=False)
plt.show()
chart_seasonal('CL', save=False)
plt.show()

**CL Notes:**

- Trend structure:
- Recent price action (any unusual moves?):
- Seasonal bias:
- Combined read:

---

## 9. ZC — Corn

In [ ]:
chart_price_daily('ZC', save=False)
plt.show()
chart_price_weekly('ZC', save=False)
plt.show()
chart_seasonal('ZC', save=False)
plt.show()

**ZC Notes:**

- Trend structure (basing or breaking out?):
- Seasonal bias (planting season effect?):
- Combined read:

---

## 10. Weekly Summary — The Big Picture

Fill this in after reviewing both notebooks. This becomes the skeleton of your Substack post.

| Market | Trend | Seasonal | COT | Overall Read |
|--------|-------|----------|-----|-------------|
| NQ | | | | |
| GC | | | | |
| CL | | | | |
| ZC | | | | |

**Top story this week:**

_What's the most interesting thing you saw? What chart would stop the scroll on X?_

## 11. Export All Charts

In [ ]:
for sym in SYMBOLS:
    chart_price_daily(sym, save=True)
    chart_price_weekly(sym, save=True)
    chart_seasonal(sym, save=True)
    plt.close('all')

print(f"Exported {len(SYMBOLS) * 3} charts to output/charts/")
print("  - price_ma_{sym}.png (daily price + MAs)")
print("  - price_weekly_{sym}.png (weekly price + OI)")
print("  - seasonal_{sym}.png (seasonal returns)")